# ML-04 — Search Intelligence Data Contract

One row is one pseudonymized content item. The feature window ends before its future outcome window, so the result is a decision-support signal rather than a causal claim.

## 1. Unit of analysis + time window

- **Grain:** One pseudonymized content item evaluated at the end of the feature window.
- **Tables:** `dim_content` provides the content-item key/context; `fact_content_daily_performance` provides daily observed performance. Join on `content_hash_id`.
- **Feature window:** The March 2026 month partition (`month=2026-03`, March 1–31). The decision moment is the end of March 31.
- **Target:** A future-window decline: April 1–30 impressions are lower than the preceding 30-day baseline (March 2–31). Keep only items with 30 observed days in both periods and a positive baseline.
- **Sealed test:** June 2026 stays untouched by this notebook for a later sealed test; these cells read only March and April.

The DuckDB check below materializes just the March–April dates needed for this contract.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("HF_TOKEN"):
    raise RuntimeError("Set HF_TOKEN in the environment or a local .env file before running this notebook.")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

from datetime import date

FEATURE_START = date(2026, 3, 1)
FEATURE_END_EXCLUSIVE = date(2026, 4, 1)
BASELINE_START = date(2026, 3, 2)
TARGET_START = date(2026, 4, 1)
TARGET_END_EXCLUSIVE = date(2026, 5, 1)
SEALED_TEST_MONTH = "2026-06"

assert (FEATURE_END_EXCLUSIVE - FEATURE_START).days == 31
assert (TARGET_START - BASELINE_START).days == 30
assert (TARGET_END_EXCLUSIVE - TARGET_START).days == 30
assert TARGET_START == FEATURE_END_EXCLUSIVE
print("Feature window:", FEATURE_START, "through", FEATURE_END_EXCLUSIVE)
print("Target comparison: 30 days before vs. 30 days after the decision moment")
print("Sealed test month reserved:", SEALED_TEST_MONTH)

Feature window: 2026-03-01 through 2026-04-01
Target comparison: 30 days before vs. 30 days after the decision moment
Sealed test month reserved: 2026-06


## 2. Fields: feature / label / context / excluded

**Features (five, all knowable at the March 31 decision moment):**

- `log1p_impressions_mar`: March search impressions, on a log scale — knowable at the decision moment because it uses only observed March feature-window impressions.
- `log1p_clicks_mar`: March search clicks, on a log scale — knowable at the decision moment because it uses only observed March feature-window clicks.
- `avg_position_mar`: impression-weighted average search position — knowable at the decision moment because it is calculated only from March daily positions and impressions.
- `ctr_pct`: March clicks divided by March impressions, expressed as a percentage — knowable at the decision moment because both counts are observed by March 31.
- `ctr_gap_pp`: the item's March CTR minus the median March CTR for its position band, in percentage points — knowable at the decision moment because both the item rate and benchmark use only the March feature window.

**Label:** `target_decline` is 1 when the next 30 days have fewer impressions than the preceding 30-day baseline; otherwise 0. It is an outcome, never an input feature.

**Context only:** `content_hash_id` is a pseudonymous join/group key, not a model feature. `report_date` defines windows; it is not a feature. The GA4 availability check is a coverage diagnostic, not a feature in this five-signal frame.

**Excluded:** Current-window `trend_direction` is explicitly excluded because it summarizes current-window movement, not information available before the decision, and would leak a proxy outcome. Future-window impressions and the derived future decline are also excluded from model inputs.

In [ ]:
SAFE_FEATURES = [
    "log1p_impressions_mar",
    "log1p_clicks_mar",
    "avg_position_mar",
    "ctr_pct",
    "ctr_gap_pp",
]
CONTEXT_FIELDS = {"content_hash_id", "report_date"}
EXCLUDED_FIELDS = {
    "trend_direction",
    "future_impressions",
    "leaked_future_30d_impressions",
    "leaked_future_decline",
}

assert len(SAFE_FEATURES) == 5
assert not (set(SAFE_FEATURES) & EXCLUDED_FIELDS)
print("Five allowed features:", ", ".join(SAFE_FEATURES))
print("Excluded from model input:", ", ".join(sorted(EXCLUDED_FIELDS)))

Five allowed features: log1p_impressions_mar, log1p_clicks_mar, avg_position_mar, ctr_pct, ctr_gap_pp
Excluded from model input: future_impressions, leaked_future_30d_impressions, leaked_future_decline, trend_direction


: 

## 3. Verify it with queries (grain, counts, missing values, windows)

The queries below check the requested `content_hash_id + report_date` grain in the March slice, the March row/date scope, and the rows with GA4 coverage. They then aggregate the five March-only features and build the April outcome from two complete 30-day periods. The fact queries read only the explicit March and April partition folders; June is not queried.

In [ ]:
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN to a Hugging Face READ token before running this cell.")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(
    f"CREATE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{os.getenv('HF_TOKEN')}');"
)
DIM_CONTENT_PATH = "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"

# Keep this bounded to the feature and target windows; June remains sealed.
con.execute(
    f"""
    CREATE OR REPLACE TEMP TABLE panel_facts AS
    SELECT report_date, content_hash_id, gsc_impressions, gsc_clicks,
           gsc_avg_position, ga4_data_available
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet', hive_partitioning = true)
    WHERE report_date >= DATE '{FEATURE_START.isoformat()}'
      AND report_date < DATE '{FEATURE_END_EXCLUSIVE.isoformat()}'
    UNION ALL
    SELECT report_date, content_hash_id, gsc_impressions, gsc_clicks,
           gsc_avg_position, ga4_data_available
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/data_0.parquet', hive_partitioning = true)
    WHERE report_date >= DATE '{TARGET_START.isoformat()}'
      AND report_date < DATE '{TARGET_END_EXCLUSIVE.isoformat()}'
    """
)
con.execute(
    f"""
    CREATE OR REPLACE TEMP VIEW march_facts AS
    SELECT * FROM panel_facts
    WHERE report_date >= DATE '{FEATURE_START.isoformat()}'
      AND report_date < DATE '{FEATURE_END_EXCLUSIVE.isoformat()}'
    """
)

# Query 1 (grain): a zero count means no duplicate content/day keys in March.
grain_check = con.sql(
    """
    SELECT COUNT(*) AS duplicate_key_count
    FROM (
        SELECT content_hash_id, report_date
        FROM march_facts
        GROUP BY content_hash_id, report_date
        HAVING COUNT(*) > 1
    )
    """
).df()
duplicate_key_count = int(grain_check.loc[0, "duplicate_key_count"])
print("Query 1 — duplicate content_hash_id + report_date keys:", duplicate_key_count)
assert duplicate_key_count == 0, "March facts do not satisfy the requested content/day grain."

# Query 2 (scope): exact row count and date coverage for March 2026.
march_scope = con.sql(
    """
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM march_facts
    """
).df()
print("Query 2 — March scope:")
display(march_scope)

# Query 3 (availability): count March rows with genuine GA4 coverage.
ga4_scope = con.sql(
    """
    SELECT COUNT(*) AS rows_with_ga4
    FROM march_facts
    WHERE ga4_data_available IS TRUE
    """
).df()
print("Query 3 — March rows surviving ga4_data_available IS TRUE:")
display(ga4_scope)

# Aggregate the feature window once per content item, then join its dimension key.
monthly_metrics = con.sql(
    """
    SELECT content_hash_id,
           SUM(gsc_impressions) AS impressions_mar,
           SUM(gsc_clicks) AS clicks_mar,
           SUM(gsc_avg_position * gsc_impressions)
             / NULLIF(SUM(CASE WHEN gsc_avg_position IS NOT NULL
                               THEN gsc_impressions END), 0) AS avg_position_mar,
           COUNT(DISTINCT report_date) AS observed_days_mar
    FROM march_facts
    GROUP BY content_hash_id
    """
).df()
dim_content = con.sql(
    f"SELECT content_hash_id FROM read_parquet('{DIM_CONTENT_PATH}')"
).df()
assert not dim_content["content_hash_id"].duplicated().any(), (
    "dim_content must have one row per content_hash_id."
)
feature_frame = dim_content.merge(
    monthly_metrics, on="content_hash_id", how="inner", validate="one_to_one"
)
assert len(feature_frame) == len(monthly_metrics), "Some March content keys did not join to dim_content."

feature_frame["log1p_impressions_mar"] = np.log1p(feature_frame["impressions_mar"])
feature_frame["log1p_clicks_mar"] = np.log1p(feature_frame["clicks_mar"])
feature_frame["ctr_pct"] = np.divide(
    feature_frame["clicks_mar"],
    feature_frame["impressions_mar"],
    out=np.full(len(feature_frame), np.nan),
    where=feature_frame["impressions_mar"].to_numpy() > 0,
) * 100

# A zero position means unavailable; do not treat it as the best possible rank.
position_for_band = feature_frame["avg_position_mar"].where(
    feature_frame["avg_position_mar"] > 0
)
feature_frame["position_band"] = pd.cut(
    position_for_band,
    bins=[0, 3, 10, 20, 50, np.inf],
    labels=["top_3", "page_1", "striking", "page_3_5", "deep"],
    include_lowest=True,
)
expected_ctr = feature_frame.groupby("position_band", observed=True)["ctr_pct"].transform("median")
feature_frame["ctr_gap_pp"] = feature_frame["ctr_pct"] - expected_ctr
assert set(SAFE_FEATURES).issubset(feature_frame.columns)
print("Five-feature frame rows:", len(feature_frame))
display(feature_frame[SAFE_FEATURES].head())

# Target: compare the complete 30-day periods immediately before and after March 31.
target_windows = con.sql(
    f"""
    SELECT content_hash_id,
           SUM(CASE WHEN report_date >= DATE '{BASELINE_START.isoformat()}'
                     AND report_date < DATE '{TARGET_START.isoformat()}'
                    THEN gsc_impressions ELSE 0 END) AS baseline_impressions,
           SUM(CASE WHEN report_date >= DATE '{TARGET_START.isoformat()}'
                     AND report_date < DATE '{TARGET_END_EXCLUSIVE.isoformat()}'
                    THEN gsc_impressions ELSE 0 END) AS future_impressions,
           COUNT(DISTINCT CASE WHEN report_date >= DATE '{BASELINE_START.isoformat()}'
                                AND report_date < DATE '{TARGET_START.isoformat()}'
                               THEN report_date END) AS baseline_days,
           COUNT(DISTINCT CASE WHEN report_date >= DATE '{TARGET_START.isoformat()}'
                                AND report_date < DATE '{TARGET_END_EXCLUSIVE.isoformat()}'
                               THEN report_date END) AS future_days
    FROM panel_facts
    GROUP BY content_hash_id
    """
).df()
target_frame = feature_frame.merge(
    target_windows, on="content_hash_id", how="inner", validate="one_to_one"
)
complete_target = (
    target_frame["baseline_days"].eq(30)
    & target_frame["future_days"].eq(30)
    & target_frame["baseline_impressions"].gt(0)
)
target_frame = target_frame.loc[complete_target].copy()
if target_frame.empty:
    raise RuntimeError("No content items have complete baseline and target windows with positive baseline impressions.")
target_frame["target_decline"] = (
    target_frame["future_impressions"] < target_frame["baseline_impressions"]
).astype("int8")
print("Items with complete target windows:", len(target_frame))
print("Observed future declines:", int(target_frame["target_decline"].sum()))

# The trap: use next-window data to recreate the target, then observe the fake perfect score.
leak_demo = target_frame[SAFE_FEATURES + ["target_decline", "future_impressions"]].copy()
leak_demo["leaked_future_30d_impressions"] = leak_demo["future_impressions"]
leak_demo["leaked_future_decline"] = (
    leak_demo["leaked_future_30d_impressions"] < target_frame["baseline_impressions"]
).astype("int8")
leaked_corr = leak_demo["leaked_future_decline"].corr(leak_demo["target_decline"])
fake_score = leak_demo["leaked_future_decline"].eq(leak_demo["target_decline"]).mean()
print(f"Leaked future-decline correlation with target: {leaked_corr:.3f}")
print(f"Leaked future-decline accuracy (invalid): {fake_score:.3f}")
print("This score is fake: the leaked column is computed from the same future window as the label.")

# Remove all future-window values before making a model-input table.
model_table = leak_demo.drop(
    columns=["future_impressions", "leaked_future_30d_impressions", "leaked_future_decline"]
)
assert not (set(model_table.columns) & EXCLUDED_FIELDS)
assert set(SAFE_FEATURES + ["target_decline"]).issubset(model_table.columns)
print("After drop: only the five March features and the target remain.")
display(model_table[SAFE_FEATURES].head())

Query 1 — duplicate content_hash_id + report_date keys: 0
Query 2 — March scope:


,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


Query 3 — March rows surviving ga4_data_available IS TRUE:


,rows_with_ga4
0,413966


Five-feature frame rows: 331437


,log1p_impressions_mar,log1p_clicks_mar,avg_position_mar,ctr_pct,ctr_gap_pp
0,0.000000,0.0,NaN,NaN,NaN
1,0.000000,0.0,NaN,NaN,NaN
2,0.000000,0.0,NaN,NaN,NaN
3,0.693147,0.0,9.0,0.0,0.0
4,0.000000,0.0,NaN,NaN,NaN


## 4. Data limits

**Limitation — unobserved Google algorithm updates in the April target window.** A search-system change could shift impressions across many items independently of any content change. This contract measures the resulting decline but cannot identify its cause, so any score is for review prioritization, not a causal claim about content quality.

In [ ]:
LIMITATION = (
    "Unobserved Google algorithm updates in April may change impressions independently "
    "of content changes; the score supports review prioritization, not causal attribution."
)
print(LIMITATION)

## Self-check

Before you submit, confirm each line honestly:

- [x] The contract, field buckets, queries, five-feature frame, leakage lesson, and limitation are filled in.
- [x] The notebook cells are syntax-checked; run the DuckDB cells top to bottom with `HF_TOKEN` set and Hugging Face dataset access enabled.
- [x] No client names, URLs, or private queries appear in the notebook.
- [x] Claims use careful words: observed, measured, directional, and decision-support.
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.